# Darcy equation: exercise 4

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k=I$ the matrix permeability and $f$ a scalar source term, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = 0\\
\nabla \cdot {q} = f
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ \nu \cdot q = 0 \text{ on } \partial \Omega$$
Where $f = \pm 1$ represents two wells, one with $+1$ and one with $-1$, set on the two corners of the domain.
The problem clearly does not have a unique solution and we need to design a strategy to make it solvable.

This is the guided ("fill in the code") version of `ex4.ipynb` -- work through the cells in order, replacing each `# TODO` with your own implementation. Compare against `ex4.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, since we will use a Raviart-Thomas approximation for ${q}$ we are restricted to simplices. In this example we consider a 2-dimensional grid.

In [ ]:
# TODO: create a 2d grid with pg.unit_grid(dim, mesh_size, as_mdg=False)
# (try mesh_size = 0.05 to start), then call sd.compute_geometry()


Let us declare the finite element spaces that we are going to use

In [ ]:
# TODO: declare the RT0 (for q) and PwConstants/P0 (for p) discretization
# objects under a key of your choice, e.g. key = "flow"
#
# TODO: build the degrees-of-freedom array
# dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])


With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: select the two well cells GEOMETRICALLY (do not hardcode cell indices --
# that breaks the moment the grid/mesh changes). For each of the two target
# corners, e.g. (0, 0) and (1, 1), find the cell whose center
# (sd.cell_centers[:2, :]) is closest to it (np.argmin of the distance)
#
# TODO: set an isotropic, unitary permeability tensor with pp.SecondOrderTensor
# and pack it into a data dictionary with pp.initialize_data (see pg.SECOND_ORDER_TENSOR)
#
# TODO: build scalar_source (length sd.num_cells): +1 at the source well cell,
# -1 at the sink well cell, 0 elsewhere
#
# TODO: the whole boundary is a natural (flux) condition here, so bc_ess is all False


Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
0\\ 
f
\end{array}
\right)
$$<br>
To construct the saddle-point problem, we rely on the `scipy.sparse` function `block_array`. Once the matrix is created, we also construct the right-hand side containing the source term.

In [ ]:
# TODO: assemble the local matrices -- the RT0 mass matrix A (with data), the P0 mass
# matrix, and the divergence matrix B = mass_p0 @ rt0.assemble_diff_matrix(sd)
#
# TODO: assemble the saddle-point matrix spp with scipy.sparse.block_array
# ([[A, -B.T], [B, None]], format="csc")


However, the matrix `spp` is singular but we can impose that the pressure has zero average
$$
    \int_\Omega p = 0 \quad \Rightarrow \quad \sum_i p_i = 0
$$
since our pressure degrees of freedom already includes the measure of the cells. We can use a Lagrange multiplier to impose this constraint, we add a new line to the system and its corresponding (anti)transpose.

In [ ]:
# TODO: build the constraint row cons (shape (1, dofs.sum())) so that cons @ [q, p]
# equals sum(p) -- it should be 1 on every pressure dof and 0 on every flux dof
#
# TODO: add the constraint to spp as an extra row/column (Lagrange multiplier):
# spp = sps.block_array([[spp, -cons.T], [cons, None]], format="csc")
#
# TODO: assemble rhs (length dofs.sum() + 1): scalar_source in the pressure block,
# 0 for the Lagrange multiplier row


We need to solve the linear system and extract the two solutions $q$ and $p$, by remembering to discard the last row used for the Lagrange multiplier.

In [ ]:
# TODO: extend bc_ess with one extra False for the Lagrange multiplier row
#
# TODO: build a pg.LinearSystem from spp and rhs, flag the essential boundary dofs,
# solve(), and discard the last entry (the Lagrange multiplier) of the solution
#
# TODO: split what remains into q and p, e.g. with
# idx = np.cumsum(dofs[:-1]); q, p = np.split(x, idx)


Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# TODO: project q to cell centers with rt0.eval_at_cell_centers(sd), and evaluate p
# at cell centers with p0.eval_at_cell_centers(sd)
#
# TODO: export cell_p and cell_q with pp.Exporter(sd, "sol", folder_name="ex4").write_vtu(...)


In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(cell_p), 0.00015528953981672954)
assert np.isclose(np.linalg.norm(cell_q), 0.009705132398026632)